In [3]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================

# OpenPIV
from openpiv import windef
from openpiv import tools, scaling, validation, filters, preprocess
import openpiv.pyprocess as process
from openpiv import pyprocess

# General
import numpy as np
import tifffile as tif
import matplotlib.pyplot as plt
import pandas as pd
from roifile import ImagejRoi

import pathlib
from time import time
import warnings
%matplotlib qt

# Custom functions
from src.PIV import run_PIV_on_frames

In [ ]:
# Integrate 

vid_3_u_list = []
vid_3_v_list = []

for current_frame in range(1, 32):
    
    # Load 
    
    vid_3_path = '/mnt/crunch/Clark/Fly_TFM/data/first/reference-time_resolved/' + f'PIVlab_{current_frame:04d}.txt'
    vid_3_data = pd.read_csv(vid_3_path, skiprows=2)
    
    # print(path)
    
    x_piv = vid_3_data['x [px]'].values
    y_piv = vid_3_data['y [px]'].values
    u_piv = vid_3_data['u [px/frame]'].values
    v_piv = vid_3_data['v [px/frame]'].values
    
    vid_3_u_list.append(u_piv)
    vid_3_v_list.append(v_piv)

u_cumulative = np.cumsum(vid_3_u_list, axis=0)  # shape (31, n_points) - running total per frame
v_cumulative = np.cumsum(vid_3_v_list, axis=0)

u_total = u_cumulative[-1] 
v_total = v_cumulative[-1]

In [46]:
# Load manual centerline for now
roi = ImagejRoi.fromfile('/mnt/crunch/Clark/Fly_TFM/data/first/0001-1216-0912.roi')
centerline_points = roi.coordinates() # numpy array of (x, y)

plt.figure(figsize=(10, 10))
plt.quiver(x_piv, y_piv, u_total, v_total)
plt.plot(centerline_points[:, 0], centerline_points[:, 1], 'r-o')
plt.axis('equal')
plt.show()

In [ ]:
# reference
u_ref_list = []
v_ref_list = []
for frame in range(1, 33):
    path = f'/mnt/crunch/Clark/Fly_TFM/data/first/reference-350/PIVlab_{frame:04d}.txt'
    
    data = pd.read_csv(path, skiprows=2)
    x = data['x [px]'].values
    y = data['y [px]'].values
    u = data['u [px/frame]'].values
    v = data['v [px/frame]'].values
    
    u_ref_list.append(u)
    v_ref_list.append(v)
u_ref = np.array(u_ref_list)
v_ref = np.array(v_ref_list)

# integrate
u_tr_list = [np.zeros_like(u_ref_list[0])] 
v_tr_list = [np.zeros_like(v_ref_list[0])]
for frame in range(1, 32):
    path = f'/mnt/crunch/Clark/Fly_TFM/data/first/reference-time_resolved/PIVlab_{frame:04d}.txt'
    
    data = pd.read_csv(path, skiprows=2)
    x = data['x [px]'].values
    y = data['y [px]'].values
    u = data['u [px/frame]'].values
    v = data['v [px/frame]'].values
    
    u_tr_list.append(u)
    v_tr_list.append(v)
u_tr = np.cumsum(u_tr_list, axis=0)
v_tr = np.cumsum(v_tr_list, axis=0)

# Compare
residual_u = u_ref - u_tr
residual_v = v_ref - v_tr
rms_per_point_per_frame = np.sqrt(residual_u**2 + residual_v**2)

In [85]:
print(u_tr.shape)
print(v_tr.shape)
print(u_ref.shape)
print(v_ref.shape)

(32, 3969)
(32, 3969)
(32, 3969)
(32, 3969)


In [87]:
import napari
import numpy as np

# reshape into grid
nx = len(np.unique(x))
ny = len(np.unique(y))

def to_grid(vals_1d):
    return vals_1d.reshape(ny, nx)

n_frames = u_tr.shape[0]  # should be 32

u_tr_grid = np.array([to_grid(u_tr[k]) for k in range(n_frames)])
v_tr_grid = np.array([to_grid(v_tr[k]) for k in range(n_frames)])
mag_tr_grid = np.sqrt(u_tr_grid**2 + v_tr_grid**2)

# napari
viewer = napari.Viewer()
viewer.add_image(mag_tr_grid, name='magnitude', colormap='jet')

yy, xx = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')

vectors = np.zeros((n_frames, ny, nx, 2, 3))
for k in range(n_frames):
    vectors[k, :, :, 0, 0] = k
    vectors[k, :, :, 0, 1] = yy
    vectors[k, :, :, 0, 2] = xx
    vectors[k, :, :, 1, 1] = v_tr_grid[k]
    vectors[k, :, :, 1, 2] = u_tr_grid[k]

vectors = vectors.reshape(-1, 2, 3)
viewer.add_vectors(vectors, name='displacement', edge_color='black', length=1, edge_width=0.5)

napari.run()

In [93]:
n_points = u_ref.shape[1]

corr_u = np.array([np.corrcoef(u_ref[:, p], u_tr[:, p])[0, 1] for p in range(n_points)])
corr_v = np.array([np.corrcoef(v_ref[:, p], v_tr[:, p])[0, 1] for p in range(n_points)])

corr_u_grid = to_grid(corr_u)
corr_v_grid = to_grid(corr_v)

fig1, ax1 = plt.subplots()
im1 = ax1.imshow(corr_u_grid, cmap='coolwarm', vmin=-1, vmax=1)
ax1.set_title('correlation: u component')
plt.colorbar(im1, ax=ax1)
plt.show()

fig2, ax2 = plt.subplots()
im2 = ax2.imshow(corr_v_grid, cmap='coolwarm', vmin=-1, vmax=1)
ax2.set_title('correlation: v component')
plt.colorbar(im2, ax=ax2)
plt.show()

In [92]:
import napari
import numpy as np

# reshape reference into grid (same nx, ny, to_grid as before)
n_frames_ref = u_ref.shape[0]

u_ref_grid = np.array([to_grid(u_ref[k]) for k in range(n_frames_ref)])
v_ref_grid = np.array([to_grid(v_ref[k]) for k in range(n_frames_ref)])
mag_ref_grid = np.sqrt(u_ref_grid**2 + v_ref_grid**2)

viewer2 = napari.Viewer()
viewer2.add_image(mag_ref_grid, name='magnitude_ref', colormap='jet', contrast_limits=[0, mag_ref_grid.max()])

yy, xx = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')

vectors_ref = np.zeros((n_frames_ref, ny, nx, 2, 3))
for k in range(n_frames_ref):
    vectors_ref[k, :, :, 0, 0] = k
    vectors_ref[k, :, :, 0, 1] = yy
    vectors_ref[k, :, :, 0, 2] = xx
    vectors_ref[k, :, :, 1, 1] = v_ref_grid[k]
    vectors_ref[k, :, :, 1, 2] = u_ref_grid[k]

vectors_ref = vectors_ref.reshape(-1, 2, 3)
viewer2.add_vectors(vectors_ref, name='displacement_ref', edge_color='black', length=1, edge_width=0.1)

napari.run()